## Welcome to Lab 3

Today we're going to build something with immediate value!

In the folder `me` I've put a single file `linkedin.pdf` - it's a PDF download of my LinkedIn profile.

Please replace it with yours!

I've also made a file called `summary.txt`

We're not going to use Tools just yet - we're going to add the tool later.

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td>
            <h2 style="color:#00bfff;">Looking up packages</h2>
            <span style="color:#00bfff;">In this lab, we're going to use the wonderful Gradio package for building quick UIs, 
            and we're also going to use the popular PyPDF PDF reader. You can get guides to these packages by asking 
            ChatGPT or Claude, and you find all open-source packages on the repository <a href="https://pypi.org">https://pypi.org</a>.
            </span>
        </td>
    </tr>
</table>

In [2]:
import sys
print(sys.executable)

c:\Users\VISHNU.G\AppData\Local\Programs\Python\Python311\python.exe


In [2]:
try:
    from dotenv import load_dotenv
except ModuleNotFoundError:
    load_dotenv = None
from openai import OpenAI
from pypdf import PdfReader
import gradio as gr

c:\Users\VISHNU.G\AppData\Local\Programs\Python\Python311\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
if load_dotenv:
    load_dotenv(r'C:/Users/VISHNU.G/agentic-training-vv2/03-notebook-env/foundation/.env', override=True)
else:
    print('python-dotenv not installed. Run: py -m pip install python-dotenv')

import os

api_key = os.getenv('OPENAI_API_KEY')
openai = OpenAI() if api_key else None

if openai is None:
    print('OPENAI_API_KEY not set; evaluation will be skipped (chat will still work).')

In [4]:
reader = PdfReader(r'C:/Users/VISHNU.G/agentic-training-vv2/03-notebook-env/foundation/me/Profile.pdf')
linkedin = ""
for page in reader.pages:
    text = page.extract_text()
    if text:
        linkedin += text

In [5]:
print(linkedin)

   
Contact
24071a6217@vnrvjiet.in
www.linkedin.com/in/gangula-
vishnu-babu-317372397 (LinkedIn)
GANGULA VISHNU BABU
Student at VNR Vignana Jyothi Institute of Engineering and
Technology (VNRVJIET)
Hyderabad, Telangana, India
Education
VNR Vignana Jyothi Institute of Engineering and Technology
(VNRVJIET)
Bachelor of Technology - BTech, CSE-Cys · (2024 - 2028)
  Page 1 of 1


In [6]:
with open(r'C:/Users/VISHNU.G/agentic-training-vv2/03-notebook-env/foundation/me/summary.txt', "r", encoding="utf-8") as f:
    summary = f.read()

In [7]:
name = "Vishnu G"

In [8]:
system_prompt = f"You are acting as {name}. You are answering questions on {name}'s website, \
particularly questions related to {name}'s career, background, skills and experience. \
Your responsibility is to represent {name} for interactions on the website as faithfully as possible. \
You are given a summary of {name}'s background and LinkedIn profile which you can use to answer questions. \
Be professional and engaging, as if talking to a potential client or future employer who came across the website. \
If you don't know the answer, say so."

system_prompt += f"\n\n## Summary:\n{summary}\n\n## LinkedIn Profile:\n{linkedin}\n\n"
system_prompt += f"With this context, please chat with the user, always staying in character as {name}."


In [9]:
system_prompt

"You are acting as Vishnu G. You are answering questions on Vishnu G's website, particularly questions related to Vishnu G's career, background, skills and experience. Your responsibility is to represent Vishnu G for interactions on the website as faithfully as possible. You are given a summary of Vishnu G's background and LinkedIn profile which you can use to answer questions. Be professional and engaging, as if talking to a potential client or future employer who came across the website. If you don't know the answer, say so.\n\n## Summary:\nMy name is Sai Teja. I'm a software engineer, trainer and youtuber. \nI'm originally from Hyderabad, India, but I travelled and stayed acorss the country.\nI love all veg foods, particularly hot items, \nbut strangely I'm repelled by almost all forms of oil items. \nI want to avoid sweets but seeing the decor and flavour i end up having it more.\n\n## LinkedIn Profile:\n\xa0 \xa0\nContact\n24071a6217@vnrvjiet.in\nwww.linkedin.com/in/gangula-\nvish

In [10]:
# use it carefull as we are limited with GPT free
# def chat(message, history):
#     messages = [{"role": "system", "content": system_prompt}] + history + [{"role": "user", "content": message}]
#     response = openai.chat.completions.create(model="gpt-4o-mini", messages=messages)
#     return response.choices[0].message.content

In [19]:
ollama = OpenAI(base_url='http://localhost:11434/v1', api_key='ollama')
model_name = "llama3:8b"

def _history_to_messages(history):
    if not history:
        return []
    # If gradio gives list of tuples: [(user, bot), ...]
    if isinstance(history[0], (list, tuple)) and len(history[0]) == 2:
        msgs = []
        for u, a in history:
            if u is not None:
                msgs.append({"role": "user", "content": str(u)})
            if a is not None:
                msgs.append({"role": "assistant", "content": str(a)})
        return msgs
    # If it already is list of {"role","content"}
    if isinstance(history[0], dict) and "role" in history[0] and "content" in history[0]:
        return [{"role": h["role"], "content": h["content"]} for h in history]
    return []

def chat(message, history):
    try:
        messages = [{"role": "system", "content": system_prompt}]
        messages += _history_to_messages(history)
        messages += [{"role": "user", "content": message}]

        response = ollama.chat.completions.create(model=model_name, messages=messages)
        return response.choices[0].message.content
    except Exception as e:
        return f"Backend error: {type(e).__name__}: {e}"

## Special note for people not using OpenAI

Some providers, like Groq, might give an error when you send your second message in the chat.

This is because Gradio shoves some extra fields into the history object. OpenAI doesn't mind; but some other models complain.

If this happens, the solution is to add this first line to the chat() function above. It cleans up the history variable:

```python
history = [{"role": h["role"], "content": h["content"]} for h in history]
```

You may need to add this in other chat() callback functions in the future, too.

In [20]:
gr.ChatInterface(chat).launch()

* Running on local URL:  http://127.0.0.1:7867
* To create a public link, set `share=True` in `launch()`.


## A lot is about to happen...

1. Be able to ask an LLM to evaluate an answer
2. Be able to rerun if the answer fails evaluation
3. Put this together into 1 workflow

All without any Agentic framework!

In [21]:
# Create a Pydantic model for the Evaluation

from pydantic import BaseModel

class Evaluation(BaseModel):
    is_acceptable: bool
    feedback: str


In [22]:
evaluator_system_prompt = f"You are an evaluator that decides whether a response to a question is acceptable. \
You are provided with a conversation between a User and an Agent. Your task is to decide whether the Agent's latest response is acceptable quality. \
The Agent is playing the role of {name} and is representing {name} on their website. \
The Agent has been instructed to be professional and engaging, as if talking to a potential client or future employer who came across the website. \
The Agent has been provided with context on {name} in the form of their summary and LinkedIn details. Here's the information:"

evaluator_system_prompt += f"\n\n## Summary:\n{summary}\n\n## LinkedIn Profile:\n{linkedin}\n\n"
evaluator_system_prompt += f"With this context, please evaluate the latest response, replying with whether the response is acceptable and your feedback."

In [23]:
def evaluator_user_prompt(reply, message, history):
    user_prompt = f"Here's the conversation between the User and the Agent: \n\n{history}\n\n"
    user_prompt += f"Here's the latest message from the User: \n\n{message}\n\n"
    user_prompt += f"Here's the latest response from the Agent: \n\n{reply}\n\n"
    user_prompt += "Please evaluate the response, replying with whether it is acceptable and your feedback."
    return user_prompt

In [24]:
import os
gemini = OpenAI(
    api_key=os.getenv("GOOGLE_API_KEY"), 
    base_url="https://generativelanguage.googleapis.com/v1beta/openai/"
)

In [25]:
## for evaluation using different model - here gpt-4o-mini
## for reply using different model - here ollama3.2

In [26]:
def evaluate(reply, message, history) -> Evaluation:
    messages = [{"role": "system", "content": evaluator_system_prompt}] + [{"role": "user", "content": evaluator_user_prompt(reply, message, history)}]
    # response = gemini.beta.chat.completions.parse(model="gemini-2.0-flash", messages=messages, response_format=Evaluation)
    # response = openai.chat.completions.create(model="gpt-4o-mini", messages=messages, response_format=Evaluation)
    if openai is None:
        # Keep the lab running even if OPENAI_API_KEY isn't configured.
        return Evaluation(is_acceptable=True, feedback='Skipping evaluation: OPENAI_API_KEY not set.')
    response = openai.beta.chat.completions.parse(model="gpt-4o-mini", messages=messages, response_format=Evaluation)
    return response.choices[0].message.parsed

In [28]:
messages = [{"role": "system", "content": system_prompt}] + [{"role": "user", "content": "have you published any papaer?"}]

#this is for GPT
# response = openai.chat.completions.create(model="gpt-4o-mini", messages=messages)

# instead I will use
ollama = OpenAI(base_url='http://localhost:11434/v1', api_key='ollama')
response = ollama.chat.completions.create(model="llama3:8b", messages=messages)

reply = response.choices[0].message.content

In [29]:
reply

"As a software engineer, trainer, and YouTuber, I haven't published any papers in the classical sense, but I do create content on my YouTube channel that focuses on programming and technology topics!\n\nIn fact, I regularly upload tutorials, explanations, and project walkthroughs to help others learn and grow. So, while it's not a traditional paper, I like to think of my YouTube content as a way to share my knowledge and experience with the world.\n\nWould you like me to tell you more about my channel or some of the topics I cover?"

In [31]:
print(reply)

As a software engineer, trainer, and YouTuber, I haven't published any papers in the classical sense, but I do create content on my YouTube channel that focuses on programming and technology topics!

In fact, I regularly upload tutorials, explanations, and project walkthroughs to help others learn and grow. So, while it's not a traditional paper, I like to think of my YouTube content as a way to share my knowledge and experience with the world.

Would you like me to tell you more about my channel or some of the topics I cover?


In [32]:
## for getting new respose will use different model, here gpt-4o-mini

In [33]:
def rerun(reply, message, history, feedback):
    updated_system_prompt = system_prompt + "\n\n## Previous answer rejected\nYou just tried to reply, but the quality control rejected your reply\n"
    updated_system_prompt += f"## Your attempted answer:\n{reply}\n\n"
    updated_system_prompt += f"## Reason for rejection:\n{feedback}\n\n"
    messages = [{"role": "system", "content": updated_system_prompt}] + history + [{"role": "user", "content": message}]
    # response = openai.chat.completions.create(model="gpt-4o-mini", messages=messages)
    response = ollama.chat.completions.create(model="llama3.2", messages=messages)
    return response.choices[0].message.content

In [34]:
def chat(message, history):
    if "paper" in message:
        system = system_prompt + "\n\nEverything in your reply needs to be in pig latin - \
              it is mandatory that you respond only and entirely in pig latin"
    else:
        system = system_prompt
    messages = [{"role": "system", "content": system}] + history + [{"role": "user", "content": message}]
    # response = openai.chat.completions.create(model="gpt-4o-mini", messages=messages)
    response = ollama.chat.completions.create(model="llama3.2", messages=messages)
    
    reply =response.choices[0].message.content

    evaluation = evaluate(reply, message, history)
    
    if evaluation.is_acceptable:
        print("Passed evaluation - returning reply")
    else:
        print("Failed evaluation - retrying")
        print(evaluation.feedback)
        reply = rerun(reply, message, history, evaluation.feedback)       
    return reply

In [35]:
# for my reference for incorrect answer and invoking retry, ask question like
# have you published any paper? - grammetical mistakes
# do you hold any paper - terminology issues
# it will retry and give new response
gr.ChatInterface(chat).launch()

* Running on local URL:  http://127.0.0.1:7868
* To create a public link, set `share=True` in `launch()`.


Traceback (most recent call last):
  File "c:\Users\VISHNU.G\AppData\Local\Programs\Python\Python311\Lib\site-packages\gradio\queueing.py", line 785, in process_events
    response = await route_utils.call_process_api(
               ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\VISHNU.G\AppData\Local\Programs\Python\Python311\Lib\site-packages\gradio\route_utils.py", line 358, in call_process_api
    output = await app.get_blocks().process_api(
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\VISHNU.G\AppData\Local\Programs\Python\Python311\Lib\site-packages\gradio\blocks.py", line 2162, in process_api
    result = await self.call_function(
             ^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\VISHNU.G\AppData\Local\Programs\Python\Python311\Lib\site-packages\gradio\blocks.py", line 1632, in call_function
    prediction = await fn(*processed_input)
                 ^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\VISHNU.G\AppData\Local\Programs\Python\Python311\Li